# Predicting Professional Burnout Risk — Logit Model
**Master 1 Data Science — Université Paris-Est Créteil | 2024–2025**  
Supervised by Sylvain Chareyron

---
## Pipeline Overview
1. Data loading & exploration
2. Data cleaning & variable selection
3. MBI score construction (Maslach Burnout Inventory)
4. Feature engineering & dummy encoding
5. Logistic regression model
6. Model evaluation & results

> **Data:** `ao_panel_2016.csv` — French working conditions survey (restricted to employed workers)  
> Place the CSV file in the same directory as this notebook before running.

## 1. Imports & Data Loading

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
from sklearn.metrics import roc_auc_score, classification_report, confusion_matrix
import pickle

# Load dataset (place ao_panel_2016.csv in the same directory)
data = pd.read_csv('ao_panel_2016.csv', sep=';', low_memory=False)
print(f"Dataset shape: {data.shape}")
data.head()

## 2. Data Cleaning & Variable Selection

In [ ]:
# --- Missing values analysis ---
def missing_summary(df, threshold=None):
    missing = df.isnull().sum()
    pct = (missing / len(df)) * 100
    summary = pd.DataFrame({'Missing': missing, 'Percent': pct})
    summary = summary[summary['Missing'] > 0].sort_values('Percent', ascending=False)
    if threshold:
        summary = summary[summary['Percent'] >= threshold]
    return summary

print(missing_summary(data, threshold=50))

In [ ]:
# --- Select relevant variables from Excel file ---
variables_exc = pd.read_excel('Variables à garder.xlsx', sheet_name='Sheet1')
variables_a_garder = variables_exc['A garder'].dropna().unique()
variables_a_garder = [v.strip() for v in variables_a_garder]

# Keep only variables present in the dataset
data_cl = data[[col for col in variables_a_garder if col in data.columns]]
print(f"Selected variables: {data_cl.shape[1]} / {len(variables_a_garder)} requested")

In [ ]:
# --- Drop variables with too many missing values or low relevance ---
to_drop = [
    # >80% missing
    'TPS_INTERIM', 'CLASS', 'ETUDIPL', 'FORMINIT', 'RP10', 'HHTOT',
    'RP7A', 'RP7D', 'RP8', 'RP7E', 'RDET', 'ANARRIV', 'ARRET',
    'AUTENT', 'NBNUIT', 'TXTPPB', 'RAISTP', 'TINAP', 'TCHOCP',
    'TSANP', 'TCHOLP', 'RPB3D', 'RPB3J', 'RPB3F', 'RPB3A', 'RPB3C',
    'RPB3B', 'RPB3I', 'RPB3H', 'RPB2B', 'RPB3G', 'RPB3E', 'RPB2A',
    'OBJATTEIN', 'TPMAISON', 'ATMAL', 'TITPUBR', 'RESTMAIN', 'NBSALENTC',
    # Low relevance
    'FORTMOD1', 'MUTE', 'EVA', 'AIDCHEF', 'v1tcdd', 'EVACRIT',
    'DICHEF', 'CHGTCOLL', 'REUNION', 'MEDECIN', 'COLLECT',
    'LIEUW', 'CLASSIF', 'v1tinterim', 'TENSION4', 'HSUPCOMP',
    'CWDEBOU', 'CWVUE', 'DOMEST', 'MAISON', 'MISSION',
    'ETUDES', 'lienmig', 'TYPEMPLOI', 'FONCTION', 'NBSALENTC',
    'REUNION', 'FORTMOD1', 'revmensc', 'v1tcdd', 'RP35',
    'v1dempro', 'v1tchoc', 'v1tina', 'v1tsan', 'v1tchol'
]

data_cl = data_cl.drop(columns=[c for c in to_drop if c in data_cl.columns])

# --- Restrict to employed workers only ---
data_cl = data_cl[data_cl['SITUA'] == 1]

# --- Drop rows with missing sex, age, income ---
data_cl = data_cl.dropna(subset=['SEXE', 'AGE'])

# --- Clean income: remove missing and extreme values (D1–D99) ---
data_cl = data_cl[data_cl['revmensc'].notna() & (data_cl['revmensc'] > 0)]
q1 = data_cl['revmensc'].quantile(0.01)
q99 = data_cl['revmensc'].quantile(0.99)
data_cl = data_cl[(data_cl['revmensc'] >= q1) & (data_cl['revmensc'] <= q99)]

# --- Clean age: remove extreme values ---
q1_age = data_cl['AGE'].quantile(0.01)
q99_age = data_cl['AGE'].quantile(0.99)
data_cl = data_cl[(data_cl['AGE'] >= q1_age) & (data_cl['AGE'] <= q99_age)]

print(f"Final dataset shape: {data_cl.shape}")
print(missing_summary(data_cl, threshold=5))

## 3. Sample Description

In [ ]:
print("=== Sample Demographics ===")
print(f"N = {len(data_cl):,}")
print(f"Women: {(data_cl['SEXE'] == 2).mean():.1%} | Men: {(data_cl['SEXE'] == 1).mean():.1%}")
print(f"Mean age: {data_cl['AGE'].mean():.1f}")
print(f"Median monthly income: {data_cl['revmensc'].median():.0f} €")

# Income distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(data_cl['revmensc'], bins=50, kde=True, ax=axes[0], color='steelblue')
axes[0].set_title('Monthly income distribution')
axes[0].set_xlabel('Income (€)')

sns.histplot(data_cl['AGE'], bins=30, kde=True, ax=axes[1], color='steelblue')
axes[1].set_title('Age distribution')
axes[1].set_xlabel('Age')

plt.tight_layout()
plt.show()

## 4. MBI Score Construction (Maslach Burnout Inventory)

The burnout indicator is reconstructed from **22 binary items** across 3 sub-scores:
- **Emotional exhaustion** (9 items)
- **Depersonalization** (5 items)  
- **Reduced personal accomplishment** (8 items, reverse-coded)

Cases defined as **severe** when global score ≥ 3rd quartile (Q3 = 7).

In [ ]:
# --- Helper functions ---

def recode_binary(x, yes_vals=[1], no_vals=[2]):
    """Generic binary recoding: yes_vals → 1, no_vals → 0, else NaN"""
    if x in yes_vals:
        return 1
    elif x in no_vals:
        return 0
    return np.nan

def recode_frequency(x, high=[1, 2], low=[3, 4]):
    """Recode frequency scale: high exposure → 1, low → 0, else NaN"""
    if x in high:
        return 1
    elif x in low:
        return 0
    return np.nan

In [ ]:
# --- Emotional exhaustion (9 items) ---

# Q01: Feeling emotionally drained
data_cl['emotionnellement_vide'] = data_cl["MIN3EDM_d"]

# Q02: Feeling worn out at end of day
data_cl['a_bout_fin_journee'] = data_cl["MIN4TAG_c"]

# Q03: Feeling tired in the morning
data_cl['fatigue_matinale'] = data_cl['RPC1D'].apply(
    lambda x: 1 if x in [4, 5, 6] else (0 if x in [1, 2, 3] else np.nan)
)

# Q04-Q06: Emotional strain (calming others, distress, conflict)
for col, new_col in [('CALMER', 'CALMER_b'), ('DETRESSE', 'DETRESSE_b'), ('CONFLIT', 'CONFLIT_b')]:
    data_cl[new_col] = data_cl[col].apply(lambda x: recode_binary(x))

data_cl['effort_relationnel'] = data_cl[['CALMER_b', 'DETRESSE_b', 'CONFLIT_b']].apply(
    lambda row: 1 if row.sum(skipna=True) >= 2 else (0 if row.notna().any() else np.nan), axis=1
)

# Q07: Out-of-hours contact
data_cl['RPA2A_b'] = data_cl['RPA2A'].apply(recode_frequency)
data_cl['HSUP_b'] = data_cl['HSUP'].apply(recode_frequency)
data_cl['JOINDRE_b'] = data_cl['JOINDRE'].apply(lambda x: recode_binary(x))

data_cl['sollicitation_hors_horaires'] = data_cl[['RPA2A_b', 'HSUP_b', 'JOINDRE_b']].apply(
    lambda row: 1 if row.sum(skipna=True) >= 2 else (0 if row.notna().any() else np.nan), axis=1
)

# Q08-Q09: Frustration, feeling overwhelmed
for col in ['RP22P', 'RPA2I', 'RPA2H']:
    data_cl[f'{col}_b'] = data_cl[col].apply(recode_frequency)

data_cl['frustration_travail'] = data_cl[['RP22P_b', 'RPA2I_b', 'RPA2H_b']].apply(
    lambda row: 1 if row.sum(skipna=True) >= 2 else (0 if row.notna().any() else np.nan), axis=1
)

data_cl['MIN4TAG_b'] = data_cl['MIN4TAG'].apply(lambda x: recode_binary(x))
data_cl['RPA2B_b'] = data_cl['RPA2B'].apply(recode_frequency)
data_cl['au_bout_du_rouleau'] = data_cl.apply(
    lambda row: 1 if row['MIN4TAG_b'] == 1 and row['RPA2B_b'] == 1
    else (0 if pd.notna(row['MIN4TAG_b']) and pd.notna(row['RPA2B_b']) else np.nan), axis=1
)

# Emotional exhaustion score
exhaustion_items = ['emotionnellement_vide', 'a_bout_fin_journee', 'fatigue_matinale',
                    'effort_relationnel', 'sollicitation_hors_horaires',
                    'frustration_travail', 'au_bout_du_rouleau']
data_cl['score_epuisement'] = data_cl[exhaustion_items].sum(axis=1, skipna=True)

print(f"Emotional exhaustion score — mean: {data_cl['score_epuisement'].mean():.2f}")

In [ ]:
# --- Depersonalization (5 items) ---

# Contact stress
data_cl['EMOTION_b'] = data_cl['EMOTION'].apply(
    lambda x: 1 if x in [1, 2, 3] else (0 if x == 4 else np.nan)
)
data_cl['TENSION1_b'] = data_cl['TENSION1'].apply(lambda x: recode_binary(x))
data_cl['PUBLIC_b'] = data_cl['PUBLIC'].apply(lambda x: recode_binary(x))

data_cl['stress_contact'] = data_cl.apply(
    lambda row: 1 if row['PUBLIC_b'] == 1 and
    row[['TENSION1_b', 'DETRESSE_b', 'EMOTION_b']].sum(skipna=True) >= 2
    else (0 if row['PUBLIC_b'] == 1 and row[['TENSION1_b', 'DETRESSE_b', 'EMOTION_b']].notna().any()
          else np.nan), axis=1
)

# Desensitization
data_cl['MIN2EDM_b'] = data_cl['MIN2EDM'].apply(lambda x: recode_binary(x))
data_cl['insensibilite'] = data_cl['MIN2EDM_b']

# Dehumanization
for col in ['RP22J', 'RP22N', 'RP22O']:
    data_cl[f'{col}_b'] = data_cl[col].apply(recode_frequency)

data_cl['RP22K_b'] = data_cl['RP22K'].apply(
    lambda x: 1 if x in [1, 2, 3] else (0 if x == 4 else np.nan)
)
data_cl['deshumanisation'] = data_cl[['RP22J_b', 'RP22K_b', 'RP22N_b', 'RP22O_b']].apply(
    lambda row: 1 if row.sum(skipna=True) >= 2 else (0 if row.notna().any() else np.nan), axis=1
)

# Depersonalization score
depersonalization_items = ['stress_contact', 'insensibilite', 'deshumanisation']
data_cl['score_deshumanisation'] = data_cl[depersonalization_items].sum(axis=1, skipna=True)

print(f"Depersonalization score — mean: {data_cl['score_deshumanisation'].mean():.2f}")

In [ ]:
# --- Reduced Personal Accomplishment (8 items, reverse-coded) ---
# High score = low accomplishment = burnout risk

data_cl['MIN5EDM_b'] = data_cl['MIN5EDM'].apply(lambda x: recode_binary(x, yes_vals=[2], no_vals=[1]))
data_cl['MIN6EDM_b'] = data_cl['MIN6EDM'].apply(lambda x: recode_binary(x, yes_vals=[2], no_vals=[1]))
data_cl['MIN8EDM_b'] = data_cl['MIN8EDM'].apply(lambda x: recode_binary(x, yes_vals=[2], no_vals=[1]))

data_cl['BIENETR1_b'] = data_cl['BIENETR1'].apply(
    lambda x: 0 if x in [1, 2] else (1 if x in [3, 4] else np.nan)
)
data_cl['BIENETR2_b'] = data_cl['BIENETR2'].apply(
    lambda x: 0 if x in [1, 2] else (1 if x in [3, 4] else np.nan)
)

accomplishment_items = ['MIN5EDM_b', 'MIN6EDM_b', 'MIN8EDM_b', 'BIENETR1_b', 'BIENETR2_b']
data_cl['score_accomplissement'] = data_cl[accomplishment_items].sum(axis=1, skipna=True)

print(f"Reduced accomplishment score — mean: {data_cl['score_accomplissement'].mean():.2f}")

In [ ]:
# --- Global MBI score & binary target variable ---

data_cl['score_burnout_global'] = (
    data_cl['score_epuisement'] +
    data_cl['score_deshumanisation'] +
    data_cl['score_accomplissement']
)

print("=== Global Burnout Score ===")
print(data_cl['score_burnout_global'].describe())

# Severe burnout: score >= Q3
q3 = data_cl['score_burnout_global'].quantile(0.75)
data_cl['burnout_severe'] = (data_cl['score_burnout_global'] >= q3).astype(int)

print(f"\nQ3 threshold: {q3}")
print(f"Severe burnout prevalence: {data_cl['burnout_severe'].mean():.1%}")
print(f"  Women: {data_cl.loc[data_cl['SEXE']==2, 'burnout_severe'].mean():.1%}")
print(f"  Men:   {data_cl.loc[data_cl['SEXE']==1, 'burnout_severe'].mean():.1%}")

# Score distribution
plt.figure(figsize=(10, 5))
sns.histplot(data_cl['score_burnout_global'], bins=20, kde=True, color='steelblue')
plt.axvline(q3, color='red', linestyle='--', label=f'Q3 = {q3:.0f} (severe threshold)')
plt.title('Global Burnout Score Distribution (MBI)')
plt.xlabel('Score')
plt.legend()
plt.tight_layout()
plt.show()

## 5. Feature Engineering

In [ ]:
# --- Gender (0 = woman, 1 = man) ---
data_cl['sexe'] = data_cl['SEXE'].apply(lambda x: 1 if x == 1 else 0)

# --- Income categories ---
def categorise_income(x):
    if x <= 1350: return '≤ 1350'
    elif x <= 1700: return '1351–1700'
    elif x <= 2250: return '1701–2250'
    elif x <= 3000: return '2251–3000'
    else: return '> 3000'

data_cl['revmensc_tranche'] = data_cl['revmensc'].apply(categorise_income)
income_dummies = pd.get_dummies(data_cl['revmensc_tranche'], prefix='revmensc_tranche', drop_first=True)
data_cl = pd.concat([data_cl, income_dummies], axis=1)

# --- Education level ---
data_cl['niv_diplome_reg'] = data_cl['DIPLOME'].apply(
    lambda x: 0 if x in [10, 20] else (1 if x in [30, 31] else (2 if x in [40, 41, 42] else np.nan))
)

# --- Work-life balance ---
data_cl['CVFVP_reg'] = data_cl['CVFVP'].apply(
    lambda x: 1 if x in [1, 2] else (0 if x in [3, 4] else np.nan)
)

# --- Social support ---
data_cl['RP1_reg'] = data_cl['RP1'].apply(lambda x: recode_binary(x))

# --- Hostile behaviors at work ---
data_cl['RPB1E_b'] = data_cl['RPB1E'].apply(lambda x: recode_binary(x))  # Degrading tasks
data_cl['RPB1H_b'] = data_cl['RPB1H'].apply(lambda x: recode_binary(x))  # Prevented from speaking
data_cl['RPB1J_b'] = data_cl['RPB1J'].apply(lambda x: recode_binary(x))  # Mockery
data_cl['RPB5E_b'] = data_cl['RPB5E'].apply(
    lambda x: 1 if x in [1, 2] else (0 if x in [3, 4] else np.nan)       # Boredom
)

# --- Autonomy ---
data_cl['INITIAT_reg'] = data_cl['INITIAT']  # Keep original 1–4 scale
data_cl['IDEE_reg'] = data_cl['IDEE'].apply(recode_frequency)
data_cl['QUANTI_reg'] = data_cl['QUANTI'].apply(recode_frequency)
data_cl['OBJECTIF_reg'] = data_cl['OBJECTIF'].apply(lambda x: recode_binary(x))

# Interaction: quantified objectives × initiative
data_cl['INT_OBJECTIF_INITIAT'] = ((data_cl['OBJECTIF_reg'] == 1) & (data_cl['INITIAT_reg'] == 1)).astype(int)

# --- Relationships ---
data_cl['TENSION2_reg'] = data_cl['TENSION2'].apply(lambda x: recode_binary(x))
data_cl['ACCHEF_reg'] = data_cl['ACCHEF'].map({1: 3, 2: 2, 3: 1, 4: 0})
data_cl['AIDCOLL_reg'] = data_cl['AIDCOLL'].apply(lambda x: recode_binary(x))
data_cl['INFOCONF_reg'] = data_cl['INFOCONF'].map({1: 3, 2: 2, 3: 1, 4: 0})
data_cl['RP4B_reg'] = data_cl['RP4B'].apply(lambda x: recode_binary(x))
data_cl['JOINEXT_reg'] = data_cl['JOINEXT'].apply(lambda x: recode_binary(x))

# --- Schedule predictability (dummies) ---
data_cl['PREVIS_mois'] = (data_cl['PREVIS'] == 1).astype(int)
data_cl['PREVIS_semaine'] = (data_cl['PREVIS'] == 2).astype(int)
data_cl['PREVIS_veille'] = (data_cl['PREVIS'] == 3).astype(int)
data_cl['PREVIS_non'] = (data_cl['PREVIS'] == 4).astype(int)

# --- Employment type (dummies) ---
typemploi_dummies = pd.get_dummies(data_cl['TYPEMPLOI'], prefix='TYPEMPLOI').drop(columns=['TYPEMPLOI_1'], errors='ignore')
data_cl = pd.concat([data_cl, typemploi_dummies], axis=1)

print("Feature engineering complete.")

## 6. Logistic Regression Model

In [ ]:
# --- Define features ---
features = [
    'sexe', 'AGE', 'niv_diplome_reg',
    'CVFVP_reg', 'RP1_reg', 'RP4B_reg',
    'RPB1E_b', 'RPB1J_b', 'RPB5E_b',
    'AIDCOLL_reg', 'JOINEXT_reg', 'INFOCONF_reg', 'ACCHEF_reg',
    'PREVIS_mois', 'PREVIS_semaine', 'PREVIS_veille', 'PREVIS_non',
    'INITIAT_reg', 'IDEE_reg', 'QUANTI_reg',
    'TENSION2_reg', 'OBJECTIF_reg', 'INT_OBJECTIF_INITIAT',
    'revmensc_tranche_1351–1700', 'revmensc_tranche_1701–2250',
    'revmensc_tranche_2251–3000', 'revmensc_tranche_> 3000',
    'TYPEMPLOI_2', 'TYPEMPLOI_3', 'TYPEMPLOI_4',
    'TYPEMPLOI_5', 'TYPEMPLOI_6', 'TYPEMPLOI_7'
]

# --- Prepare model dataset ---
model_data = data_cl[features + ['burnout_severe']].dropna()
X = model_data[features]
y = model_data['burnout_severe']

# Class imbalance correction via weights
class_weight = len(y) / (2 * np.bincount(y))
weights = y.map({0: class_weight[0], 1: class_weight[1]})

print(f"Model dataset: {len(model_data):,} observations")
print(f"Burnout prevalence: {y.mean():.1%}")

# --- Fit logistic regression ---
X_const = sm.add_constant(X)
model = sm.Logit(y, X_const)
result = model.fit(freq_weights=weights, maxiter=200)
print(result.summary())

## 7. Model Evaluation & Results

In [ ]:
# --- Predictions ---
THRESHOLD = 0.20  # Low threshold to maximize recall (prevention-oriented)
y_proba = result.predict(X_const)
y_pred = (y_proba >= THRESHOLD).astype(int)

# --- Performance metrics ---
auc = roc_auc_score(y, y_proba)
print(f"AUC: {auc:.3f}")
print(f"\nClassification Report (threshold = {THRESHOLD}):")
print(classification_report(y, y_pred))

# --- ROC Curve ---
from sklearn.metrics import roc_curve
fpr, tpr, _ = roc_curve(y, y_proba)
plt.figure(figsize=(7, 5))
plt.plot(fpr, tpr, color='steelblue', label=f'AUC = {auc:.3f}')
plt.plot([0, 1], [0, 1], 'k--')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve — Burnout Logit Model')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# --- Odds Ratios ---
odds_ratios = pd.DataFrame({
    'OR': np.exp(result.params),
    'CI_low': np.exp(result.conf_int()[0]),
    'CI_high': np.exp(result.conf_int()[1]),
    'p_value': result.pvalues
}).drop('const').sort_values('OR', ascending=False)

print("=== Top Risk Factors (OR > 1) ===")
print(odds_ratios[odds_ratios['OR'] > 1].head(10).round(3))

print("\n=== Top Protective Factors (OR < 1) ===")
print(odds_ratios[odds_ratios['OR'] < 1].tail(8).round(3))

# Forest plot
sig = odds_ratios[odds_ratios['p_value'] < 0.05].sort_values('OR')
plt.figure(figsize=(8, 10))
plt.barh(sig.index, sig['OR'], color=['#d73027' if x > 1 else '#4575b4' for x in sig['OR']])
plt.axvline(1, color='black', linewidth=0.8, linestyle='--')
plt.xlabel('Odds Ratio')
plt.title('Significant Predictors (p < 0.05)')
plt.tight_layout()
plt.show()

In [ ]:
# --- Save model ---
with open('logit_model_final.pkl', 'wb') as f:
    pickle.dump(result, f)

print("Model saved to logit_model_final.pkl")